In [1]:
#import libraries

import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
import patsy
import warnings

warnings.filterwarnings("ignore")

df = pd.read_csv('retail_dynamics_weekly_sales.csv')

ModuleNotFoundError: No module named 'statsmodels'

# Task 1

## Structure Analysis

get general overview of the dataset's structure

In [ ]:
df.info()

inspect the first 5 rows for each variable to glean how data is represented in the dataset

In [ ]:
df.head(5)

quick summary of dataset size i.e. row/column count

In [ ]:
df.shape

## Content Analysis

#### Frequency analysis 

In [ ]:
display_freq = ["Display"]

for row in display_freq:
    print(df[display_freq].value_counts(dropna=False))

#### Unique identifer check

In [ ]:
print("Unique StoreIDs:", df["StoreID"].nunique())
print("Duplicate StoreIDs:", df["StoreID"].duplicated().sum())
print(df["StoreID"].nunique() == len(df))

#### Duplicate record check

check for duplicate records in dataset

In [ ]:
df.duplicated().sum()

#### Invalid Data Analysis

check for invalid strings in categorical columns

In [ ]:
invalid_str = df[['Display', 'StoreID']].isin(["?", ""])
print(invalid_str.sum())

check for nulls

In [ ]:
df.isnull().sum()

## Descriptive Statistics 

#### Statistical Summary for numerical columns

In [ ]:
df.describe(include='number').round(2)

Examine sales specifically

In [ ]:
df['Sales'].describe().round(2)

### Sales behaviour analysis

Distribution of Sales

In [ ]:
df['Sales'].hist()

sales by store

In [ ]:
Store_sales = df.groupby('StoreID')['Sales'].mean()

Store_sales.describe().round(2)

Calculate stores with top 5 lowest vs highest sales

In [ ]:
print("Lowest average sales:")
print(Store_sales.nsmallest(5))

print("\nHighest average sales:")
print(Store_sales.nlargest(5))

##### Sales under Display vs No display

In [ ]:
df.groupby('Display')['Sales'].mean().round(2)

#### Sales under Media vs No Media

In [ ]:
media_sales = df.groupby('Media')['Sales'].mean().round(2)

print("No Media:", media_sales[0])
print("Media:", media_sales[1])

#### Sales under Higher price vs lower price

find median price to establish higher/lower boundary cutoff

In [ ]:
price_median = df['Price'].median()

print("Median Price:", price_median)

create a temporary price level category 

In [ ]:
df['Price_Level'] = np.where(
    df['Price'] >= price_median,
    'Higher Price',
    'Lower Price'
)

check that worked

In [ ]:
df['Price_Level'].value_counts()

#### avg sales at higher vs lower price level

In [ ]:
price_sales = df.groupby('Price_Level')['Sales'].mean().round(2)
price_sales

In [ ]:
df['Display'].value_counts()

#### Sales comparison

In [ ]:
display = df['Display'].astype('string').str.strip().str.casefold()

display_yes = df.loc[display.isin(['yes', 'y']), 'Sales'].mean()
display_no = df.loc[display.isin(['no', 'n']), 'Sales'].mean()

comparison = pd.DataFrame({
    'Condition': [
        'Display',
        'No Display',
        'Media',
        'No Media',
        'Higher Price',
        'Lower Price'
    ],
    'Average Sales': [
        display_yes,
        display_no,
        media_sales[1],
        media_sales[0],
        price_sales['Higher Price'],
        price_sales['Lower Price']
    ]
})

comparison

In [ ]:
comparison.plot(
    x='Condition',
    y='Average Sales',
    kind='bar',
    legend=False,
    title='Average Weekly Sales by Promotional Condition',
    ylabel='Average Sales (units)'
)

# Task 2

## Data Cleansing

#### Mapping

standardise different values as per frequency analysis

In [ ]:
df['Display'] = df['Display'].str.strip().str.lower()

Binary encode Display such that No = 0 and Yes = 1

In [ ]:
df['Display'] = df['Display'].replace({
    'no': 0,
    'n': 0,
    'yes': 1,
    'y': 1
})

Check encoding

In [ ]:
df['Display'].value_counts(dropna=False)

In [ ]:
df.info()

## Regression Model